# ClaimLens: AI-generated-image detector (GenImage)

Trains the detector `services/guardrails` uses to score claim photos.

1. Mount Google Drive (your GenImage subset lives there).
2. Train on **Stable Diffusion v1.4** only; test on the val split of **every** generator (GenImage's protocol, so most test generators are unseen).
3. Features: a frozen **CLIP ViT-B/16** image encoder. Classifier: **logistic regression** on those features (the UnivFD recipe).
4. Pick the threshold for a **1% false-positive rate** on real photos: a false flag sends an honest customer to an investigator.
5. Export `clip_visual.onnx` + `probe.json` to Drive, for `data/models/ai-detector/`.

**The JPEG shortcut.** GenImage's real images are ImageNet JPEGs and its fakes are PNGs. A detector can score
well just by noticing JPEG compression, which fails on the first JPEG fake (every phone upload is JPEG). So every
image, real or generated, is re-encoded as JPEG at a random quality and size before CLIP sees it. Section 4 shows
the format split; section 7 (optional) trains without the fix to show what it hides.

**Runtime:** Runtime > Change runtime type > **T4 GPU**. With the defaults, feature extraction takes roughly 15-25 minutes.

**What you need in Drive:** for SD v1.4 the `train/{ai,nature}` and `val/{ai,nature}` folders, and for each other
generator at least `val/{ai,nature}`, in GenImage's layout (`<generator>/<generator>/{train,val}/{ai,nature}`). You
don't need all 1.3M images: a few thousand per class for training and a few hundred per generator for testing is
plenty for a linear probe.

**Before running:** push the repo (this notebook clones it to import `training/ai_detector.py` and the service's
preprocessing). **Data licence:** GenImage is for research use; don't redistribute it.

## 1. Config

In [ ]:
# Edit these to match your Drive.
GENIMAGE_ROOT = "/content/drive/MyDrive/GenImage"
OUT_DIR = "/content/drive/MyDrive/claimlens/ai-detector"
REPO_URL = "https://github.com/AndrewRaj0510/ClaimLens.git"
REPO_DIR = "/content/ClaimLens"

CLIP_MODEL = "openai/clip-vit-base-patch16"
N_TRAIN = 4000  # images per class from SD v1.4 train
N_TEST = 500  # images per class per generator, from val
HOLDOUT = 0.1  # share of the training images kept back to set the threshold
MAX_FPR = 0.01  # at most 1% of real photos flagged
BATCH = 64
SEED = 0
SHORTCUT_EXPERIMENT = True  # section 7: also train on un-aligned images, for comparison

In [ ]:
!nvidia-smi

## 2. Drive, repo, packages

In [ ]:
import os
import sys

from google.colab import drive

drive.mount("/content/drive")
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
sys.path.insert(0, f"{REPO_DIR}/training")
!pip install -q onnx onnxruntime

## 3. Find the generators

In [ ]:
import numpy as np

import ai_detector as ad

gens = ad.find_generators(GENIMAGE_ROOT)
for g, splits in gens.items():
    sizes = {
        s: {k: len(ad.list_images(v)) for k, v in d.items()} for s, d in splits.items()
    }
    print(f"{g:40s} {sizes}")
train_gen = next(g for g in gens if ad.TRAIN_GENERATOR in g.lower())
test_gens = [g for g in gens if "val" in gens[g]]
print("train on:", train_gen, "| test on:", test_gens)

## 4. Sample, and look at the file formats

In [ ]:
train = {
    label: ad.sample(ad.list_images(gens[train_gen]["train"][label]), N_TRAIN, SEED)
    for label in ("ai", "nature")
}
test = {
    g: {
        label: ad.sample(ad.list_images(gens[g]["val"][label]), N_TEST, SEED)
        for label in ("ai", "nature")
    }
    for g in test_gens
}
print("train formats:", ad.format_report(train))
for g in test_gens:
    print(f"{g:40s}", ad.format_report(test[g]))

## 5. CLIP features (images aligned: JPEG at a random quality and size)

In [ ]:
import torch
from transformers import CLIPVisionModelWithProjection

device = "cuda" if torch.cuda.is_available() else "cpu"
clip = CLIPVisionModelWithProjection.from_pretrained(CLIP_MODEL).to(device).eval()


@torch.no_grad()
def encode(x):
    return (
        clip(pixel_values=torch.from_numpy(x).to(device))
        .image_embeds.float()
        .cpu()
        .numpy()
    )


def features(paths, aligned=True, seed=SEED):
    return ad.extract(paths, encode, batch_size=BATCH, seed=seed, aligned=aligned)


X_ai, X_real = features(train["ai"]), features(train["nature"], seed=SEED + 1)
F_test = {
    g: (features(t["ai"], seed=SEED + 2), features(t["nature"], seed=SEED + 3))
    for g, t in test.items()
}
print(X_ai.shape, X_real.shape)

## 6. Train the probe, set the threshold, evaluate

In [ ]:
rng = np.random.default_rng(SEED)


def split(X):
    idx = rng.permutation(len(X))
    k = max(1, int(len(X) * HOLDOUT))
    return X[idx[k:]], X[idx[:k]]


fit_ai, hold_ai = split(X_ai)
fit_real, hold_real = split(X_real)
X = np.r_[fit_ai, fit_real]
y = np.r_[np.ones(len(fit_ai)), np.zeros(len(fit_real))]
clf = ad.fit_probe(X, y, seed=SEED)
w, b = clf.coef_[0], float(clf.intercept_[0])
print("C =", clf.C_[0])

threshold = ad.pick_threshold(ad.probe_scores(hold_real, w, b), MAX_FPR)
results = {
    "holdout_" + train_gen: ad.metrics(
        ad.probe_scores(hold_ai, w, b), ad.probe_scores(hold_real, w, b), threshold
    )
}
for g, (fa, fr) in F_test.items():
    results[g] = ad.metrics(
        ad.probe_scores(fa, w, b), ad.probe_scores(fr, w, b), threshold
    )
unseen = [m for g, m in results.items() if g in F_test and g != train_gen]
summary = (
    {
        k: round(float(np.mean([m[k] for m in unseen])), 4)
        for k in ("auc", "accuracy", "tpr", "fpr")
    }
    if unseen
    else {}
)

print(f"threshold {threshold:.4f} (target FPR {MAX_FPR})")
print(f"{'set':40s} {'AUC':>6s} {'acc':>6s} {'TPR':>6s} {'FPR':>6s}")
for g, m in results.items():
    print(
        f"{g:40s} {m['auc']:6.3f} {m['accuracy']:6.3f} {m['tpr']:6.3f} {m['fpr']:6.3f}"
    )
print("mean over unseen generators:", summary)

## 7. (Optional) What the JPEG shortcut hides

Same probe, trained on the original files (PNG fakes, JPEG reals), then tested on aligned images. If its numbers
collapse while section 6's hold up, the un-aligned detector had learned "PNG means fake".

In [ ]:
if SHORTCUT_EXPERIMENT:
    n = min(1000, N_TRAIN)
    Xs = np.r_[
        features(train["ai"][:n], aligned=False),
        features(train["nature"][:n], aligned=False),
    ]
    ys = np.r_[np.ones(n), np.zeros(n)]
    raw = ad.fit_probe(Xs, ys, seed=SEED)
    rw, rb = raw.coef_[0], float(raw.intercept_[0])
    raw_thr = ad.pick_threshold(ad.probe_scores(Xs[n:], rw, rb), MAX_FPR)
    for g, (fa, fr) in F_test.items():
        m = ad.metrics(
            ad.probe_scores(fa, rw, rb), ad.probe_scores(fr, rw, rb), raw_thr
        )
        print(f"{g:40s} trained un-aligned, tested aligned: {m}")

## 8. Export the encoder to ONNX and check it matches PyTorch

In [ ]:
import random

import onnxruntime as ort


class Visual(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, pixel_values):
        return self.model(pixel_values=pixel_values).image_embeds


os.makedirs(OUT_DIR, exist_ok=True)
onnx_path = f"{OUT_DIR}/clip_visual.onnx"
cpu_model = Visual(CLIPVisionModelWithProjection.from_pretrained(CLIP_MODEL).eval())
torch.onnx.export(
    cpu_model,
    (torch.zeros(1, 3, 224, 224),),
    onnx_path,
    input_names=["pixel_values"],
    output_names=["image_embeds"],
    dynamic_axes={"pixel_values": {0: "batch"}, "image_embeds": {0: "batch"}},
    opset_version=17,
    dynamo=False,
)

# The same images through PyTorch and ONNX Runtime, preprocessed by the service's own code.
sample_paths = test[test_gens[0]]["ai"][:4] + test[test_gens[0]]["nature"][:4]
x = ad.load_batch(sample_paths, random.Random(SEED))
with torch.no_grad():
    ref = cpu_model(torch.from_numpy(x)).numpy()
got = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"]).run(
    None, {"pixel_values": x}
)[0]


def unit(f):
    return f / np.linalg.norm(f, axis=1, keepdims=True)


diff = float(np.abs(unit(ref) - unit(got)).max())
score_diff = float(
    np.abs(ad.probe_scores(unit(ref), w, b) - ad.probe_scores(unit(got), w, b)).max()
)
print(f"max feature diff {diff:.2e}, max score diff {score_diff:.2e}")
assert diff < 1e-3 and score_diff < 1e-3

## 9. Save the probe

In [ ]:
probe = ad.probe_json(
    w,
    b,
    threshold,
    {
        "per_set": results,
        "mean_unseen": summary,
        "train_generator": train_gen,
        "n_train_per_class": N_TRAIN,
        "n_test_per_class": N_TEST,
        "max_fpr": MAX_FPR,
    },
    model_name=f"{CLIP_MODEL} + logistic probe (GenImage {train_gen})",
    output_name="image_embeds",
    notes="Images JPEG-aligned before features; threshold set for the target FPR on held-out real images.",
)
ad.save_probe(f"{OUT_DIR}/probe.json", probe)
print(os.listdir(OUT_DIR), f"{os.path.getsize(onnx_path) / 1e6:.0f} MB")

## 10. Use it

1. Download `clip_visual.onnx` and `probe.json` from `claimlens/ai-detector/` in Drive into
   `data/models/ai-detector/` (git-ignored).
2. `docker compose restart guardrails`; its `/health` then says `"ai_detector": true`.
3. Copy the table from section 6 into `docs/metrics.md`.

**Caveat to state with the numbers:** GenImage is ImageNet content (animals, objects), not cars. How well the probe
does on AI-made car-damage photos is untested until there's a set of those.